In [0]:
# Challenge 5 Setup: Designing a Dashboard-Ready Dataset
# Creates a star schema for students to build gold views and metric views from.

import re
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DateType, DoubleType
from datetime import date, timedelta
import random

# --- Schema setup ---
username = spark.sql("SELECT current_user()").collect()[0][0]
clean_username = re.sub(r'[^a-zA-Z0-9]', '_', username.split('@')[0])
catalog = "main"
schema = f"challenge_5_{clean_username}"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS `{catalog}`.`{schema}`")
spark.sql(f"USE CATALOG `{catalog}`")
spark.sql(f"USE SCHEMA `{schema}`")

print(f"Schema: {catalog}.{schema}")

In [0]:
# --- dim_stores: 10 rows ---
spark.sql("DROP TABLE IF EXISTS dim_stores")
spark.sql("""
  CREATE TABLE dim_stores (
    store_id INT,
    store_name STRING,
    city STRING,
    state STRING,
    channel STRING
  )
""")
spark.sql("""
  INSERT INTO dim_stores VALUES
    (1, 'Downtown Flagship', 'New York', 'NY', 'retail'),
    (2, 'Mall of America', 'Minneapolis', 'MN', 'retail'),
    (3, 'Tech Hub Store', 'San Francisco', 'CA', 'retail'),
    (4, 'Outlet Center', 'Orlando', 'FL', 'outlet'),
    (5, 'Airport Express', 'Chicago', 'IL', 'retail'),
    (6, 'Web Store US', 'N/A', 'N/A', 'online'),
    (7, 'Web Store Canada', 'N/A', 'N/A', 'online'),
    (8, 'Mobile App', 'N/A', 'N/A', 'online'),
    (9, 'Partner Portal', 'N/A', 'N/A', 'wholesale'),
    (10, 'Warehouse Direct', 'Dallas', 'TX', 'wholesale')
""")
print("Created dim_stores: 10 rows")

In [0]:
# --- dim_customers: 30 rows ---
spark.sql("DROP TABLE IF EXISTS dim_customers")
spark.sql("""
  CREATE TABLE dim_customers (
    customer_id INT,
    customer_name STRING,
    segment STRING,
    region STRING
  )
""")

random.seed(55)
segments = ['Enterprise', 'Mid-Market', 'SMB', 'Consumer']
regions = ['Northeast', 'Southeast', 'Midwest', 'West']
names = [
    'Acme Corp', 'Beta Systems', 'Cloud Nine', 'Delta Force', 'Echo Labs',
    'Frontier Tech', 'Global Reach', 'Horizon AI', 'Infinite Loop', 'Jade Networks',
    'Keystone Inc', 'Lunar Dynamics', 'Metro Solutions', 'Nova Partners', 'Orbit Data',
    'Pinnacle Group', 'Quantum Edge', 'Relay Services', 'Summit Ops', 'Titan Industries',
    'Unity Works', 'Vertex Systems', 'Wave Digital', 'Xcel Analytics', 'Yield Corp',
    'Zenith Labs', 'Alpha Bridge', 'Blue Shift', 'Core Logic', 'DataStream'
]

insert_values = []
for i, name in enumerate(names, 1):
    seg = segments[(i - 1) % 4]
    reg = regions[(i - 1) % 4]
    insert_values.append(f"({i}, '{name}', '{seg}', '{reg}')")

spark.sql(f"INSERT INTO dim_customers VALUES {', '.join(insert_values)}")
print("Created dim_customers: 30 rows")

In [0]:
# --- dim_products: 20 rows ---
spark.sql("DROP TABLE IF EXISTS dim_products")
spark.sql("""
  CREATE TABLE dim_products (
    product_id INT,
    product_name STRING,
    category STRING,
    subcategory STRING,
    unit_cost DOUBLE
  )
""")

products = [
    (1, 'Laptop Pro', 'Electronics', 'Computers', 450.00),
    (2, 'Tablet Air', 'Electronics', 'Computers', 200.00),
    (3, 'Wireless Mouse', 'Electronics', 'Accessories', 12.00),
    (4, 'USB-C Hub', 'Electronics', 'Accessories', 25.00),
    (5, 'Noise-Cancel Headphones', 'Electronics', 'Audio', 85.00),
    (6, 'Denim Jacket', 'Clothing', 'Outerwear', 28.00),
    (7, 'Running Shoes', 'Clothing', 'Footwear', 35.00),
    (8, 'Polo Shirt', 'Clothing', 'Tops', 12.00),
    (9, 'Winter Coat', 'Clothing', 'Outerwear', 55.00),
    (10, 'Canvas Sneakers', 'Clothing', 'Footwear', 18.00),
    (11, 'Standing Desk', 'Home', 'Furniture', 180.00),
    (12, 'Ergonomic Chair', 'Home', 'Furniture', 150.00),
    (13, 'LED Desk Lamp', 'Home', 'Lighting', 15.00),
    (14, 'Bookcase', 'Home', 'Furniture', 60.00),
    (15, 'Smart Speaker', 'Home', 'Electronics', 30.00),
    (16, 'Yoga Mat', 'Sports', 'Fitness', 8.00),
    (17, 'Dumbbell Set', 'Sports', 'Fitness', 40.00),
    (18, 'Tennis Racket', 'Sports', 'Racquet', 55.00),
    (19, 'Hiking Boots', 'Sports', 'Outdoor', 65.00),
    (20, 'Camping Tent', 'Sports', 'Outdoor', 90.00)
]

values_str = ', '.join([f"({p[0]}, '{p[1]}', '{p[2]}', '{p[3]}', {p[4]})" for p in products])
spark.sql(f"INSERT INTO dim_products VALUES {values_str}")
print("Created dim_products: 20 rows")

In [0]:
# --- fact_orders: 2000 rows ---
spark.sql("DROP TABLE IF EXISTS fact_orders")

random.seed(88)

schema_def = StructType([
    StructField("order_id", IntegerType(), False),
    StructField("order_date", DateType(), False),
    StructField("customer_id", IntegerType(), False),
    StructField("product_id", IntegerType(), False),
    StructField("store_id", IntegerType(), False),
    StructField("quantity", IntegerType(), False),
    StructField("unit_price", DoubleType(), False),
    StructField("discount_amt", DoubleType(), False)
])

rows = []
for i in range(1, 2001):
    order_date = date(2024, 1, 1) + timedelta(days=random.randint(0, 364))
    unit_price = round(random.uniform(20, 600), 2)
    qty = random.randint(1, 5)
    discount = round(random.choice([0, 0, 0, 0, unit_price * 0.05, unit_price * 0.10, unit_price * 0.15]), 2)
    rows.append((
        i,
        order_date,
        random.randint(1, 30),
        random.randint(1, 20),
        random.randint(1, 10),
        qty,
        unit_price,
        discount
    ))

df = spark.createDataFrame(rows, schema=schema_def)
df.write.saveAsTable("fact_orders")
print(f"Created fact_orders: {df.count()} rows")

In [0]:
# --- Summary ---
print("="*60)
print("CHALLENGE 5 SETUP COMPLETE")
print("="*60)
print(f"Catalog:  {catalog}")
print(f"Schema:   {schema}")
print(f"")
print(f"Star Schema (Silver layer):")
print(f"  fact_orders    - 2000 rows (order_id, order_date, customer_id, product_id, store_id, quantity, unit_price, discount_amt)")
print(f"  dim_customers  - 30 rows (customer_id, customer_name, segment, region)")
print(f"  dim_products   - 20 rows (product_id, product_name, category, subcategory, unit_cost)")
print(f"  dim_stores     - 10 rows (store_id, store_name, city, state, channel)")
print(f"")
print(f"Your tasks: Build Gold views and metric views from this star schema.")
print(f"")
print(f"Next: Open the SQL Editor and run:")
print(f"  USE CATALOG {catalog};")
print(f"  USE SCHEMA {schema};")